# 04 - Text Model Optimisation

**Goal:** find the best settings for the text model.

**Method:**
1. Grid search with 3-fold cross-validation on the training set only.
2. Compare the best LinearSVC with two other models.
3. Score the final model once on the validation set.

The validation set is not used to choose any setting.

## 1. Load the data

In [1]:
import time

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

data = pd.read_csv("../data/processed/text_train_processed.csv")
train = data[data["split"] == "train"]
validation = data[data["split"] == "validation"]

X_train, y_train = train["text_combined"], train["prdtypecode"]
X_validation, y_validation = validation["text_combined"], validation["prdtypecode"]

print(f"Train:      {len(train):,}")
print(f"Validation: {len(validation):,}")

Train:      67,932
Validation: 16,984


## 2. Grid search on LinearSVC

The 03 notebook already showed that using all features is better than 50,000 (+1.2 points with CV on train). So `max_features` is fixed at no limit.

We test 2 settings together:
- `binary`: count words (False) or only mark if they appear (True)
- `C`: how closely the SVM fits the training data (small C = simpler model)

That is 2 x 4 = 8 combinations, each scored with 3-fold CV on train.

In [2]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("svm", LinearSVC(random_state=42)),
])

param_grid = {
    "tfidf__binary": [False, True],
    "svm__C": [0.1, 0.5, 1, 2],
}

start = time.time()
grid = GridSearchCV(pipeline, param_grid, cv=3, scoring="f1_weighted", n_jobs=4)
grid.fit(X_train, y_train)
print(f"Grid search time: {(time.time() - start) / 60:.1f} min")

grid_results = pd.DataFrame(grid.cv_results_)[[
    "param_tfidf__binary", "param_svm__C", "mean_test_score", "std_test_score",
]]
grid_results.columns = ["binary", "C", "cv_weighted_f1", "cv_std"]
grid_results.sort_values("cv_weighted_f1", ascending=False).round(4).reset_index(drop=True)

Grid search time: 10.5 min


,binary,C,cv_weighted_f1,cv_std
0,True,1.0,0.8284,0.0026
1,True,2.0,0.8282,0.0020
2,True,0.5,0.8263,0.0030
3,False,1.0,0.8235,0.0025
4,False,2.0,0.8231,0.0024
5,False,0.5,0.8223,0.0025
6,True,0.1,0.8031,0.0013
7,False,0.1,0.7999,0.0017


In [3]:
# Rule: keep binary=True only if its gain is bigger than the CV spread
best_by_binary = grid_results.loc[grid_results.groupby("binary")["cv_weighted_f1"].idxmax()].set_index("binary")
binary_gain = best_by_binary.loc[True, "cv_weighted_f1"] - best_by_binary.loc[False, "cv_weighted_f1"]
cv_spread = grid_results["cv_std"].max()

use_binary = bool(binary_gain > cv_spread)
best_C = best_by_binary.loc[use_binary, "C"]

print(f"Binary gain:  {binary_gain:+.4f}")
print(f"CV spread:    {cv_spread:.4f}")
print(f"Use binary:   {use_binary}")
print(f"Best C:       {best_C}")

final_tfidf = dict(ngram_range=(1, 2), min_df=2, binary=use_binary)

Binary gain:  +0.0050
CV spread:    0.0030
Use binary:   True
Best C:       1.0


## 3. Other models

We keep the chosen TF-IDF settings and swap the classifier:
- **Complement Naive Bayes:** a fast, simple model often used for text.
- **Logistic Regression (SGD):** a linear model like the SVM, but it outputs probabilities.

Each is scored with the same 3-fold CV on train.

## 3. Other models

We keep the best TF-IDF settings and swap the classifier:
- **Complement Naive Bayes:** a fast, simple model often used for text.
- **Logistic Regression (SGD):** a linear model like the SVM, but it outputs probabilities.

Each is scored with the same 3-fold CV on train.

In [4]:
other_models = {
    "LinearSVC (chosen)": LinearSVC(C=best_C, random_state=42),
    "Complement Naive Bayes": ComplementNB(),
    "Logistic Regression (SGD)": SGDClassifier(loss="log_loss", alpha=1e-5, random_state=42),
}

model_rows = []
for name, model in other_models.items():
    pipe = Pipeline([("tfidf", TfidfVectorizer(**final_tfidf)), ("model", model)])
    start = time.time()
    scores = cross_val_score(pipe, X_train, y_train, cv=3, scoring="f1_weighted", n_jobs=3)
    model_rows.append({
        "model": name,
        "cv_weighted_f1": scores.mean(),
        "cv_std": scores.std(),
        "time_min": (time.time() - start) / 60,
    })

pd.DataFrame(model_rows).sort_values("cv_weighted_f1", ascending=False).round(4).reset_index(drop=True)

,model,cv_weighted_f1,cv_std,time_min
0,LinearSVC (chosen),0.8284,0.0026,0.7438
1,Logistic Regression (SGD),0.8084,0.0015,0.3758
2,Complement Naive Bayes,0.7600,0.0003,0.2275


### Analysis

- **C = 1 is best.** C = 0.1 is clearly worse (about -2.5 points). C = 0.5, 1 and 2 are almost equal.
- **Binary helps here.** The best binary model scores 0.8284 and the best non-binary model scores 0.8235. The gain (+0.50 points) is bigger than the CV spread (0.30), so we keep `binary=True`.
- With all features, the binary gain is clearer than in the 03 notebook (+0.33 with 50,000 features).
- **Chosen settings:** all features, `binary=True`, `C=1`.

## 4. Final score on validation

We train the baseline and the best model on the full training set, and score both once on validation.

In [5]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), min_df=2, binary=True)),
    ("svm", LinearSVC(random_state=42)),
])

optimised = Pipeline([
    ("tfidf", TfidfVectorizer(**final_tfidf)),
    ("svm", LinearSVC(C=best_C, random_state=42)),
])

final_rows = []
for name, model in [("Baseline (03 notebook)", baseline), ("Optimised", optimised)]:
    model.fit(X_train, y_train)
    pred = model.predict(X_validation)
    final_rows.append({
        "model": name,
        "weighted_f1": f1_score(y_validation, pred, average="weighted"),
        "macro_f1": f1_score(y_validation, pred, average="macro"),
    })

pd.DataFrame(final_rows).round(4)

,model,weighted_f1,macro_f1
0,Baseline (03 notebook),0.8313,0.8192
1,Optimised,0.8436,0.8330


### Analysis

- **LinearSVC is the best model** (0.8284).
- Logistic Regression is 2 points lower (0.8084). It is faster and gives probabilities, but the score loss is too big.
- Complement Naive Bayes is 7 points lower (0.7600). It is the fastest, but too simple for 27 categories.
- **We keep LinearSVC.**